Dallin: make sure you update this to do just a straight import of sbdynt as sbd and then call any functions as sbd.whatever (make it match the other examples in this directory)

In [1]:
import numpy as np
import rebound
import matplotlib.pyplot as plt
import os
import pandas as pd
from datetime import datetime

os.chdir('../src/')


import sbdynt as sbd

# Default Proper Element Run

Default properties for computing proper elements for TNOs is contained within the same ``run_tno`` function used to initialize and run the machine learning algorithm for TNO resonance occupation.
A similar function, ``run_asteroid`` can be used to similarly run and compute proper elements for asteroids. We demonstrate these functions below for TNO (15760) Albion and the dwarf planet and asteroid (1) Ceres. 

To compute proper elements as well, from the ``run_tno`` function, user should set the boolean variable ``run_proper = True``, which will compute the proper elements and save the results to the ``tno_result`` output. 

The bulk of the time required by these functions is actually spent integrating the simulation, or if the simulation is already integrated, in reading the orbital elements from the Simulation Archive binary files. The actual computation of the proper elements is comparatively fast.

In this TNO example, we will still run the machine learning algorithm by including ``run_ML = True``, as this produces a more complex Simulation Archive with varying time resolution. This demonstrates how these complex archives are automatically handled while computing the proper elements from the simulation. We also include print statements to demonstrate and benchmark how long the default runs may take to fully integrate. 

In [ ]:
begin = datetime.now()
tno_result = sbd.run_tno(des='Albion', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None, 
                                     logfile=False,deletefile=True, run_ML = True, run_proper=True)
print(datetime.now()-begin)


begin = datetime.now()
ast_result = sbd.run_ast(des='Ceres', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None,
                                     logfile=False,deletefile=True)
print(datetime.now()-begin)

Running TNO ML
Running TNO integration for PE/Stability Indicators


Functions are also provided to use an existing Simulation archive for a TNO or asteroid to compute the proper elements from that archive file, which bypasses the need to reintegrate the entire small body.

An archive file that is not computed using SBDynT may be used, but user must make sure that the archive file has the appropriate planets for the object with the correct hashes as identifiers, and must also make sure that the proper ``object_type`` is specified. However, we recommend typically implementing these functions only with archive files produced using SBDynT code, as formatting differences may occur. 

Since we have already integrated Simulation archives above for TNO ``(15760) Albion`` and asteroid ``(1) Ceres``, we may use these functions to more quickly compute the proper elements for these objects.

In [ ]:
begin = datetime.now()
tno_result = sbd.run_existing_tno(des='Albion', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None, 
                                     logfile=False,deletefile=False, run_ML = False, run_proper=True, output_arrays = True)
print(datetime.now()-begin)


begin = datetime.now()
ast_result = sbd.run_existing_sb(des='Ceres', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None, object_type='asteroid',
                                     logfile=False,deletefile=False, run_proper=True, output_arrays = True)
print(datetime.now()-begin)

# Proper Element Outputs

The proper element results are saved to a ``proper_element`` class object within the ``tno_result`` object. 
This class object contains a large number of relevant values and indicators related to the orbital evolution of the small body corresponding to the proper motion over time. These include...

```proper_elements```

```mean_elements```

```osculating_elements```

```proper_errors```

```planet_freqs```

```proper_windows```

### Proper Elements

The proper elements themselves are contained in a dictionary with the same name as the ``proper_element`` class. These can be contrasted with the mean elements, (which are simply the mean of the osculating element time array), and the initial osculating elements, (which represent the orbital elements at time ``t=0`` in the simulation).

We note that these mean elements are not be confused with the ``mean`` indicator, which is used to identify objects which experience chaotic motion or long-term periodic evolution such that the synethic proper element cannot be accurately computed for the small-body in the given integration. 
We will discuss the ``mean`` indicator, as well as other useful indicators, in more detail in the ``proper_elements_advanced`` notebook. 

These three outputs contain the semi-major axis, eccentricity, inclination, as well as the proper and mean precession rates, ``g`` and ``s``, in the case of the ``proper_elements`` and ``mean_elements`` variables. The ``osculating_elements`` variable instead reports the initial ``omega`` and ``Omega`` values at the ``t=0`` epoch.

The proper and mean ``g`` and ``s`` frequencies are reported in both revolutions/year (the units of measurement used by the filtering, the direct result from the filter) and arcseconds/year (the typically reported value for the proper precession rates).

In [ ]:
print('Albion: ',tno_result.proper_elements.proper_elements)
print('\nCeres:', ast_result.proper_elements.proper_elements)


In [ ]:
print('Albion:', tno_result.proper_elements.mean_elements)
print('\nCeres:', ast_result.proper_elements.mean_elements)

In [ ]:
print('Albion:',tno_result.proper_elements.osculating_elements)
print('\nCeres:',ast_result.proper_elements.osculating_elements)

### Proper Element Uncertainties

The proper element uncertainties are contained in the ``proper_errors`` dictionary.

In [ ]:
print('Albion:', tno_result.proper_elements.proper_errors)
print('\nCeres:',ast_result.proper_elements.proper_errors)

### Planetary Frequencies

Users interested in the identified planetary secular frequencies can retrieve these from the ``planets`` dictionary. These are only reported in the units of rev/yr. Users can retrieve the "/yr units by multipying these results by 1296000.

In [ ]:
print('Albion:', tno_result.proper_elements.planet_freqs)
print('\nCeres:', ast_result.proper_elements.planet_freqs)

### Window Proper Elements

Users may also access the proper element computed for each of the windows. This could be useful in cases of objects which experience chaotic or scattering motion, if the users wishes to see the proper element before such events. 

In [ ]:
print('Albion:', tno_result.proper_elements.proper_windows)
print('\nCeres:', ast_result.proper_elements.proper_windows)

More advanced outputs are further discussed in the ``proper_elements_advanced.ipynb`` file, which include flags related to chaos, the amplitude of secular terms, as well as some indicators which may be hekpful for identifying secualr resonance occupation. 